网格读取、几何结构生成、流场初始化

In [1]:
import meshreading as mr
import geometry as geo
import initialize as ini
import classconfig as cc
import solvesupple as ss

mr.read_mesh("fangdata.txt")
geo.geometry_main("output.txt")
ini.initialization_main()

tscell : cc.cell_class= cc.CellList[1][1]

successfully read fangdata.txt with 120 points.
i_total: 10, j_total: 12, meshcnt: 120


守恒量计算,计算的守恒量是$\boldsymbol{U} = (\rho, \rho u, \rho v, \rho E, \rho \tilde{\nu})$,使用`cc.cell_class.formvars`方法计算.


In [2]:
for i in range(1, cc.i_total,1):  # 一定要注意i的范围是什么
    for j in range(1,cc.j_total+1,1):
        cell : cc.cell_class = cc.CellList[i][j]
        cell.formvars()

# 测试tscell的守恒变量是否正确
# print(tscell.U)

基于CFL和谱半径近似的最小时间步长,公式为$\Delta t_{ij} = \frac{\text{CFL} \cdot V_{ij}}{|uA+vB| + |uC+vD| + c_{ij} \cdot (\sqrt{A^2+B^2} + \sqrt{C^2+D^2})}$,其中(A,B)和(C,D)为相邻两面法向的平均值.

In [3]:
import math
mintime = float('inf')

# 第一轮:计算各单元 localdt, 记录最小值
for j in range(1, cc.j_total + 1):
    jp1 = j + 1 if j < cc.j_total else 1          # 周向回绕
    for i in range(1, cc.i_total):
        cell : cc.cell_class= cc.CellList[i][j]
        A = 0.5 * (cc.Facelist_tau[i][j].nx+ cc.Facelist_tau[i+1][j].nx)
        B = 0.5 * (cc.Facelist_tau[i][j].ny+ cc.Facelist_tau[i+1][j].ny)
        C = 0.5 * (cc.FaceList_n[i][j].nx+ cc.FaceList_n[i][jp1].nx)
        D = 0.5 * (cc.FaceList_n[i][j].ny+ cc.FaceList_n[i][jp1].ny)
        E = abs(cell.u * A + cell.v * B)
        F = abs(cell.u * C + cell.v * D)
        G = math.sqrt(A * A + B * B)
        L = math.sqrt(C * C + D * D)
        cell.localdt = cc.CFL * cell.vol / (E + F + cell.c * (G + L))

        if cell.localdt < mintime:
            mintime = cell.localdt

# 第二轮:所有单元均使用全局最小时间步推进
for j in range(1, cc.j_total + 1):
    for i in range(1, cc.i_total):
        cc.CellList[i][j].dt = mintime

cc.totaltime += mintime
print("current time:", cc.totaltime,"min_time:", mintime)

current time: 0.00044076390910298765 min_time: 0.00044076390910298765


生成虚拟网格`Imagine_Mesh`,共有壁面虚拟网格、远场虚拟网格、成环虚拟网格三种类型

In [4]:
"""设置壁面虚拟网格,使用镜像法"""

for im in range(1, cc.IM + 1):
    ghost_row = [[]]                       # j=0 占位
    for j in range(1, cc.j_total + 1):
        gcell : cc.cell_class = cc.cell_class((cc.i_total + im - 1, j))

        # 标量: 从壁面直接复制
        gcell.rho = cc.CellList[1][j].rho
        gcell.p   = cc.CellList[1][j].p
        gcell.T   = cc.CellList[1][j].T
        gcell.E   = cc.CellList[1][j].E
        gcell.H   = cc.CellList[1][j].H
        gcell.c   = cc.CellList[1][j].c

        # 速度 / 湍流粘度: 取对应内层的相反数 (镜像反射)
        gcell.u     = -cc.CellList[im][j].u
        gcell.v     = -cc.CellList[im][j].v
        gcell.miubl = -cc.CellList[im][j].miubl
        gcell.ma = (math.sqrt(cc.CellList[im][j].u ** 2 +
                                cc.CellList[im][j].v ** 2) / cc.CellList[1][j].c)
        gcell.formvars()
        ghost_row.append(gcell)
    cc.CellList.append(ghost_row)

"""设置压力远场假想网格边界条件"""

for im in range(1, cc.IM + 1):
    ghost_row = [[]]             # j=0 占位
    for j in range(1, cc.j_total + 1):
        gcell = cc.cell_class((cc.i_total + im - 1, j))
        gcell.copy_flow_fields(cc.CellList[cc.i_total - 1][j])
        gcell.formvars()
        ghost_row.append(gcell)
    cc.CellList.append(ghost_row)

"""设置 O 型网格切割线两侧的周期假想网格 (j 方向周期边界)
左侧 ghost ← 右侧物理端 (j = j_total, j_total-1, ...)
右侧 ghost ← 左侧物理端 (j = 1, 2, ...)"""

for i in range(1, cc.i_total):
    # ── 左侧假想网格 ──
    for im in range(1, cc.IM + 1):
        gcell = cc.cell_class((i, cc.j_total + im))
        gcell.copy_flow_fields(cc.CellList[i][cc.j_total - im + 1])
        gcell.formvars()
        cc.CellList[i].append(gcell)

    # ── 右侧假想网格 ──
    for im in range(1, cc.IM + 1):
        gcell = cc.cell_class((i, cc.j_total + cc.IM + im))
        gcell.copy_flow_fields(cc.CellList[i][im])
        gcell.formvars()
        cc.CellList[i].append(gcell)

# print(vars(cc.CellList[cc.i_total+cc.IM-1][1]))

计算对流项.

1. 先在每个面上准备面上的守恒量$\boldsymbol{U_f}$,其做法是$\boldsymbol{U_f} = \frac{\boldsymbol{U_1}+\boldsymbol{U_2}}{2}$.
2. 根据这个守恒量,变化出面上对流项$\boldsymbol{F} = (\rho V_n , \rho u V_n + p n_x, \rho v V_n + p n_y, (\rho e+p)V_n,\rho \tilde{\nu} V_n)^\top$,其中$V_n = u n_x + v n_y$
3. 把所有面通量汇总到单元中心,形成对流项$\boldsymbol{F_c}$

这里我们使用了一种Gauss:$\int_V \left (\frac{\partial F}{\partial x}+\frac{\partial G}{\partial y} \right) \mathrm{d} V = \oint_{\partial V} (F,G) \cdot \vec{\boldsymbol{n}} \mathrm{d}S$

In [5]:
# 处理壁面处的面上守恒量,face_tau,由首层tau网格和第一层壁面虚拟网格平均
for j in range(1,cc.j_total+1):
    face : cc.face_class = cc.Facelist_tau[1][j]
    face.form_face_conserved_1stbounded(cc.CellList[1][j], cc.CellList[cc.i_total][j])

# 处理远场处的面上守恒量,face_tau,由最外层tau网格和第一层远场虚拟网格平均
for j in range(1,cc.j_total+1):
    face : cc.face_class = cc.Facelist_tau[cc.i_total][j]
    face.form_face_conserved_1stbounded(cc.CellList[cc.i_total-1][j], cc.CellList[cc.i_total+cc.IM][j])

# 处理左周期边界处的面上守恒量,face_n
for i in range(1,cc.i_total):
    face : cc.face_class = cc.FaceList_n[i][cc.j_total]
    face.form_face_conserved_1stbounded(cc.CellList[i][cc.j_total], cc.CellList[i][cc.j_total+1])
    
# 处理右周期边界处的面上守恒量,face_n
for i in range(1,cc.i_total):
    face : cc.face_class = cc.FaceList_n[i][1]
    face.form_face_conserved_1stbounded(cc.CellList[i][1], cc.CellList[i][cc.j_total+cc.IM+1])

# 处理正常地方的面上守恒量,时刻提醒自己face_tau是(i_total,j_total),face_n是(i_total-1,j_total)
for i in range(2,cc.i_total): # (1,j)和(i_total,j)的face通量已经处理好了
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.Facelist_tau[i][j]
        face.form_face_conserved_1stbounded(cc.CellList[i][j], cc.CellList[i-1][j])

for i in range(1,cc.i_total):
    for j in range(2,cc.j_total+1): # (i,1)和(i,j_total)的face通量已经处理好了
        face : cc.face_class = cc.FaceList_n[i][j]
        face.form_face_conserved_1stbounded(cc.CellList[i][j],cc.CellList[i][j-1])

# 现在开始构造对流项:
for i in range(1,cc.i_total+1):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.Facelist_tau[i][j]
        face.form_flux()

for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.FaceList_n[i][j]
        face.form_flux()

# 形成单元体的总通量,请注意以(i,j)变大为正.
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        cell: cc.cell_class = cc.CellList[i][j]
        jp1 = j + 1 if j < cc.j_total else 1  
        cell.Fc = (cc.FaceList_n[i][jp1].Flux-cc.FaceList_n[i][j].Flux+
                   cc.Facelist_tau[i+1][j].Flux-cc.Facelist_tau[i][j].Flux)

计算因S-A模型引起的扩散项
1. 计算梯度,使用Green-Gauss方法:$\overline{\nabla \phi} = \frac{1}{V} \oint \phi \vec{\boldsymbol{n}} \mathrm{d} S$
2. 合成扩散项$\boldsymbol{F_v}$

In [6]:
from grad import green_gauss
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        cell : cc.cell_class = cc.CellList[i][j]

        # 处理各种特殊情况
        i_down = cc.i_total if i==1 else i-1
        i_up = cc.i_total+cc.IM if i==cc.i_total-1 else i+1
        j_left = cc.j_total+1 if j==1 else j-1
        j_right = cc.j_total+cc.IM+1 if j==cc.j_total else j+1
        jpright = j + 1 if j < cc.j_total else 1

        # 找到邻接网格
        cell_up : cc.cell_class = cc.CellList[i_up][j]
        cell_down : cc.cell_class = cc.CellList[i_down][j]
        cell_left : cc.cell_class = cc.CellList[i][j_left]
        cell_right : cc.cell_class = cc.CellList[i][j_right]

        # 找到邻接面
        face_up : cc.face_class = cc.Facelist_tau[i+1][j]
        face_down : cc.face_class = cc.Facelist_tau[i][j]
        face_left : cc.face_class = cc.FaceList_n[i][j]
        face_right : cc.face_class = cc.FaceList_n[i][jpright]

        # 使用一阶中心差分建构邻接面上的物理量
        face_up.form_face_vars_1stbounded(cell,cell_up)
        face_down.form_face_vars_1stbounded(cell,cell_down)
        face_left.form_face_vars_1stbounded(cell,cell_left)
        face_right.form_face_vars_1stbounded(cell,cell_right)

        # 建立当前网格的Green-Guass梯度
        green_gauss(cell,face_up,face_down,face_right,face_left)

# 对壁面虚拟网格进行处理
for j in range(1,cc.j_total+1):
    cell : cc.cell_class = cc.CellList[cc.i_total][j]
    cell.copy_grad(cc.CellList[1][j],ifT=False)

# 对远场虚拟网格进行处理
for j in range(1,cc.j_total+1):
    cell : cc.cell_class = cc.CellList[cc.i_total+cc.IM][j]
    cell.copy_grad(cc.CellList[cc.i_total-1][j],ifu=False,ifv=False,ifT=False,ifmiubl=False)

# 对左周期处的虚拟网格进行处理
for i in range(1,cc.i_total):
    cell : cc.cell_class = cc.CellList[i][cc.j_total+1]
    cell.copy_grad(cc.CellList[i][cc.j_total])

# 对右周期处的虚拟网格进行处理
for i in range(1,cc.i_total):
    cell : cc.cell_class = cc.CellList[i][cc.j_total+cc.IM+1]
    cell.copy_grad(cc.CellList[i][1])

In [7]:
import turbulence as tb

# 计算壁面假想网格 S-A湍流模型各个参量
for j in range(1,cc.j_total+1):
    cell : cc.cell_class = cc.CellList[cc.i_total][j]
    tb.Spalart_Allmaras(cell)

# 计算远场假想网格 S-A湍流模型各个参量
for j in range(1,cc.j_total+1):
    cell : cc.cell_class = cc.CellList[cc.i_total+cc.IM][j]
    tb.Spalart_Allmaras(cell)

# 计算左周期处的虚拟网格 S-A湍流模型各个参量
for i in range(1,cc.i_total):
    cell : cc.cell_class = cc.CellList[i][cc.j_total+1]
    tb.Spalart_Allmaras(cell)

# 计算右周期处的虚拟网格 S-A湍流模型各个参量
for i in range(1,cc.i_total):
    cell : cc.cell_class = cc.CellList[i][cc.j_total+cc.IM+1]
    tb.Spalart_Allmaras(cell)

# 计算正常网格的 S-A湍流模型各个参量
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        cell : cc.cell_class = cc.CellList[i][j]
        tb.Spalart_Allmaras(cell)

# 处理facelist_tau的面上扩散项
for i in range(1,cc.i_total+1):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.Facelist_tau[i][j]
        # 处理特殊情况
        i_down = cc.i_total if i==1 else i-1
        i_up = cc.i_total + cc.IM if i==cc.i_total else i
        # 找到邻接网格
        cell_up : cc.cell_class = cc.CellList[i_up][j]
        cell_down : cc.cell_class = cc.CellList[i_down][j]
        # 形成扩散项
        tb.form_face_diffusion_1stbounded(face,cell_up, cell_down)

# 处理facelist_n的面上扩散项
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.FaceList_n[i][j]
        # 处理特殊情况
        j_left = cc.j_total+1 if j==1 else j-1
        j_right = cc.j_total+cc.IM+1 if j==cc.j_total else j
        # 找到邻接网格
        cell_left : cc.cell_class = cc.CellList[i][j_left]
        cell_right : cc.cell_class = cc.CellList[i][j_right]
        # 形成扩散项
        tb.form_face_diffusion_1stbounded(face,cell_left, cell_right)

# 形成cell的湍流扩散项
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        cell : cc.cell_class = cc.CellList[i][j]
        jp1 = j+1 if j<cc.j_total else 1
        face_up : cc.face_class = cc.Facelist_tau[i+1][j]
        face_down : cc.face_class= cc.Facelist_tau[i][j]
        face_left : cc.face_class = cc.FaceList_n[i][j]
        face_right : cc.face_class = cc.FaceList_n[i][jp1]
        # 形成湍流扩散项
        cell.Fv = (face_up.DiffuTurb - face_down.DiffuTurb+
                   face_right.DiffuTurb - face_left.DiffuTurb)

计算因S-A 湍流模型导致的源项$\boldsymbol{S}$

In [8]:
# # 首先计算在每个tau面上的湍流粘度梯度∇miubl
# for i in range(1,cc.i_total+1):
#     for j in range(1,cc.j_total+1):
#         i_down = cc.i_total if i==1 else i-1
#         i_up = cc.i_total + cc.IM if i==cc.i_total else i
#         face : cc.face_class = cc.Facelist_tau[i][j]
#         cell_up : cc.cell_class = cc.CellList[i_up][j]
#         cell_down : cc.cell_class = cc.CellList[i_down][j]
#         face.miublgrad = (cell_up.miublgrad+cell_down.miublgrad)/2

# for i in range(1,cc.i_total):
#     for j in range(1,cc.j_total+1):
#         face : cc.face_class = cc.FaceList_n[i][j]
#         j_left = cc.j_total+1 if j==1 else j-1
#         j_right = cc.j_total+cc.IM+1 if j==cc.j_total else j
#         cell_left : cc.cell_class = cc.CellList[i][j_left]
#         cell_right : cc.cell_class = cc.CellList[i][j_right]
#         face.miublgrad = (cell_left.miublgrad+cell_right.miublgrad)/2

# 构建源项S
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        cell : cc.cell_class = cc.CellList[i][j]
        tb.form_source_term(cell)

构建人工粘性$\boldsymbol{F_d}$

In [ ]:
import dissipation as hs

# tau边界上的谱半径近似
for i in range(1,cc.i_total+1):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.Facelist_tau[i][j]
        if i==1:hs.Spectral_Radius(face,cc.CellList[1][j],cc.CellList[1][j])
        elif i==cc.i_total:hs.Spectral_Radius(face,cc.CellList[cc.i_total-1][j],cc.CellList[cc.i_total-1][j])
        else: hs.Spectral_Radius(face,cc.CellList[i][j],cc.CellList[i-1][j])
        print("tau,lamdba:",face.index,face.lambda_f)

# n边界上的谱半径近似
for i in range(1,cc.i_total):
    for j in range(1,cc.j_total+1):
        face : cc.face_class = cc.FaceList_n[i][j]
        j_left = cc.j_total if j==1 else j-1
        hs.Spectral_Radius(face,cc.CellList[i][j],cc.CellList[i][j_left])
        print("n,lamdba:",face.index,face.lambda_f)

#tau边界上激波捕捉

tau,lamdba: (1, 1) 319.391773084367
tau,lamdba: (1, 2) 374.54153101911106
tau,lamdba: (1, 3) 410.90017558749935
tau,lamdba: (1, 4) 410.90017558749935
tau,lamdba: (1, 5) 374.54153101911106
tau,lamdba: (1, 6) 319.391773084367
tau,lamdba: (1, 7) 319.391773084367
tau,lamdba: (1, 8) 374.54153101911106
tau,lamdba: (1, 9) 410.90017558749935
tau,lamdba: (1, 10) 410.90017558749935
tau,lamdba: (1, 11) 374.54153101911106
tau,lamdba: (1, 12) 319.391773084367
tau,lamdba: (2, 1) 370.9966773698661
tau,lamdba: (2, 2) 421.02505581695834
tau,lamdba: (2, 3) 452.26489019326283
tau,lamdba: (2, 4) 452.26489019326283
tau,lamdba: (2, 5) 421.02505581695834
tau,lamdba: (2, 6) 370.9966773698661
tau,lamdba: (2, 7) 370.9966773698661
tau,lamdba: (2, 8) 421.02505581695834
tau,lamdba: (2, 9) 452.26489019326283
tau,lamdba: (2, 10) 452.26489019326283
tau,lamdba: (2, 11) 421.02505581695834
tau,lamdba: (2, 12) 370.9966773698661
tau,lamdba: (3, 1) 474.4419072325793
tau,lamdba: (3, 2) 514.5786635120618
tau,lamdba: (3, 3) 5